# Laboratorio 9 — Redes Neuronales Artificiales (RNA)
## CC3074 Minería de Datos · SmartStay Advisors · Semestre I 2026

**Objetivo:** Construir modelos de RNA para (1) clasificar el precio de listings Airbnb en categorías Barato / Medio / Caro y (2) predecir el precio numérico, comparando ambos tipos contra los algoritmos de laboratorios anteriores.

## 0. Carga de librerías

In [ ]:
# ─── Librerías ───────────────────────────────────────────────────────────────
import warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline

# ── Reproducibilidad ─────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Estilo visual ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print('✓ Librerías cargadas')

## 1. Carga y preparación del dataset

Se carga `listings.RData` y se replica **exactamente** el mismo preprocesamiento y la misma partición 80/20 usados en los laboratorios anteriores para que la comparación sea válida.

In [ ]:
# ─── 1.1  Leer RData (formato XDR binario comprimido con gzip) ───────────────
import gzip, re, struct, io

def parse_rdata_doubles(raw, label):
    """Encuentra la sección de doubles de una columna numérica."""
    idx = raw.find(label.encode() + b'\x00')
    if idx == -1:
        return None
    return idx

# Leer todo el archivo descomprimido
with gzip.open('data/listings.RData', 'rb') as f:
    raw = f.read()
print(f'Archivo descomprimido: {len(raw):,} bytes')

In [ ]:
# ─── 1.2  Extraer strings de columnas categóricas conocidas del bloque final ──
# Las columnas tipo character (factor/string) se almacenan como vectores de
# strings nulos-terminados en R's XDR. Usamos una función que los lee.

def extract_string_vector(raw, col_name, next_col_name=None, max_vals=50000):
    """
    Extrae un vector de strings de R a partir del marcador de la columna.
    """
    marker = (b'\x00\t\x00\x00\x00' +
              bytes([len(col_name)]) + col_name.encode())
    pos = raw.rfind(marker)
    if pos == -1:
        # fallback: buscar por longitud de 2 bytes
        marker2 = (b'\x00\x04\x00\t\x00\x00\x00' +
                   bytes([len(col_name)]) + col_name.encode())
        pos = raw.rfind(marker2)
    if pos == -1:
        return None
    return pos

# Verificar posición de columnas clave
for c in ['price', 'room_type', 'neighbourhood_cleansed', 'property_type']:
    p = raw.rfind(c.encode())
    print(f'{c}: last occurrence at byte {p:,}')

In [ ]:
# ─── 1.3  Parsear el RData completo usando una función robusta ────────────────
# Dada la complejidad del formato RDX3 binario, usamos un parser basado en
# el hecho de que los números doubles están en big-endian IEEE 754 de 8 bytes.

def read_r_double_vector(raw, start_after_marker):
    """Lee un vector REALSXP de R (tipo 14 = 0x0e) a partir de su marca de tamaño."""
    # Buscamos el patrón: tipo(4) + len(4) + datos (big-endian)
    pos = start_after_marker
    # El tipo REALSXP=14 se codifica como \x00\x00\x00\x0e
    # Saltamos a encontrar el tamaño
    size_bytes = raw[pos:pos+4]
    n = struct.unpack('>I', size_bytes)[0]
    pos += 4
    data = struct.unpack(f'>{n}d', raw[pos:pos + n*8])
    return np.array(data), pos + n*8

print('Parser definido ✓')

In [ ]:
# ─── 1.4  Construcción del DataFrame ─────────────────────────────────────────
# Estrategia: dado que el parsing binario completo requeriría un intérprete R
# completo, utilizamos el bloque de nombres al final del archivo para mapear
# columnas a sus bloques de datos y reconstruimos el dataframe.

# ── Encontrar el bloque de nombres de columnas ──
# Los nombres están cerca del final del archivo
NAMES_REGION_START = 455_845_000
tail = raw[NAMES_REGION_START:]

# Extraer nombres de columnas con regex sobre el bloque
col_pattern = re.compile(
    rb'\x00\x04\x00\t\x00\x00\x00([\x01-\x3f])([A-Za-z][A-Za-z0-9_.]{0,62})\x00'
)
col_names = []
for m in col_pattern.finditer(tail):
    length = m.group(1)[0]
    name = m.group(2).decode('ascii', errors='ignore')
    if len(name) == length:
        col_names.append(name)

# Lista manual validada (extraída de exploración binaria previa)
COLS = [
    'id','listing_url','scrape_id','last_scraped','source','name','description',
    'neighborhood_overview','picture_url','host_id','host_url','host_name',
    'host_location','host_response_time','host_response_rate','host_acceptance_rate',
    'host_is_superhost','host_thumbnail_url','host_picture_url','host_neighbourhood',
    'host_listings_count','host_total_listings_count','host_verifications',
    'host_has_profile_pic','host_identity_verified','neighbourhood',
    'neighbourhood_cleansed','neighbourhood_group_cleansed','latitude','longitude',
    'property_type','room_type','accommodates','bathrooms','bathrooms_text',
    'bedrooms','beds','amenities','price','minimum_nights','maximum_nights',
    'minimum_minimum_nights','maximum_minimum_nights','minimum_maximum_nights',
    'maximum_maximum_nights','minimum_nights_avg_ntm','maximum_nights_avg_ntm',
    'calendar_updated','has_availability','availability_30','availability_60',
    'availability_90','availability_365','calendar_last_scraped',
    'number_of_reviews','number_of_reviews_ltm','number_of_reviews_l30d',
    'availability_eoy','number_of_reviews_ly','estimated_occupancy_l365d',
    'estimated_revenue_l365d','first_review','last_review',
    'review_scores_rating','review_scores_accuracy','review_scores_cleanliness',
    'review_scores_checkin','review_scores_communication','review_scores_location',
    'review_scores_value','license','instant_bookable',
    'calculated_host_listings_count','calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms','reviews_per_month','city'
]
print(f'Columnas identificadas: {len(COLS)}')

In [ ]:
# ─── 1.5  Extraer vectores numéricos de bloques de datos ─────────────────────
# Para las columnas numéricas continuas usamos parsing directo del bloque de doubles.
# El patrón de un vector REALSXP en XDR:
#   \x00\x00\x00\x0e  (tipo 14 = REALSXP)
#   \x00\x00\x00\x0N  (flags)
#   \x00\x00\x00\xNN  (n elements, 4 bytes big-endian)
#   [n × 8 bytes, big-endian doubles]

# ── Localizar bloques de doubles ──────────────────────────────────────────────
# Estrategia: buscamos todos los bloques REALSXP de tamaño N (N = número de filas).
# Primero determinamos N del primer vector numérico encontrado.

# La columna 'latitude' es numérica pura; buscamos su bloque en la región de datos
# (primera mitad del archivo, antes de la sección de nombres)

def find_double_blocks(raw, n_expected=None, region_end=None):
    """Encuentra todos los bloques REALSXP válidos."""
    pattern = re.compile(rb'\x00\x00\x00\x0e\x00\x02\x00\x0e')
    blocks = []
    end = region_end or len(raw)
    for m in pattern.finditer(raw, 0, end):
        pos = m.start() + 8  # skip type+flags
        if pos + 4 > end:
            continue
        n = struct.unpack('>I', raw[pos:pos+4])[0]
        if n_expected and n != n_expected:
            continue
        if n < 100 or n > 2_000_000:
            continue
        data_start = pos + 4
        data_end = data_start + n * 8
        if data_end > end:
            continue
        blocks.append((m.start(), n, data_start, data_end))
    return blocks

# Buscar en los primeros 300 MB (datos)
print('Buscando bloques de doubles (puede tardar ~10s)...')
blocks = find_double_blocks(raw, region_end=300_000_000)
print(f'Bloques encontrados: {len(blocks)}')
if blocks:
    sizes = [b[1] for b in blocks]
    from collections import Counter
    most_common = Counter(sizes).most_common(5)
    print('Tamaños más frecuentes:', most_common)

In [ ]:
# ─── 1.6  Determinar N (número de filas) y extraer columnas numéricas ─────────
from collections import Counter

sizes = [b[1] for b in blocks]
N = Counter(sizes).most_common(1)[0][0]
print(f'Número de filas del dataset: {N:,}')

# Filtrar solo bloques con N filas
main_blocks = [b for b in blocks if b[1] == N]
print(f'Bloques con {N} filas: {len(main_blocks)}')

# Extraer todos los vectores numéricos
numeric_arrays = []
for (start, n, ds, de) in main_blocks:
    arr = np.array(struct.unpack(f'>{n}d', raw[ds:de]))
    numeric_arrays.append(arr)

print(f'Vectores numéricos extraídos: {len(numeric_arrays)}')

In [ ]:
# ─── 1.7  Mapear vectores a columnas por contenido ────────────────────────────
# Usamos heurísticas: latitude ~25-35, longitude ~(-100 a -70), 
# accommodates 1-16, price 10-1000, etc.

def match_col(arr, col):
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0:
        return False
    mn, mx, med = valid.min(), valid.max(), np.median(valid)
    if col == 'latitude':      return 20 < mn < 70 and 20 < mx < 70
    if col == 'longitude':     return -180 < mn < 0 and -180 < mx < 0
    if col == 'accommodates':  return 1 <= mn and mx <= 50 and med < 10 and np.all(valid == valid.astype(int))
    if col == 'bedrooms':      return 0 <= mn and mx <= 30 and med < 5
    if col == 'beds':          return 0 <= mn and mx <= 50 and med < 6
    if col == 'price':         return mn >= 0 and mx < 100000 and 10 < med < 2000
    if col == 'minimum_nights':return 1 <= mn and mx <= 10000 and med < 100
    if col == 'number_of_reviews': return mn >= 0 and mx < 10000
    if col == 'review_scores_rating': return 0 <= mn and mx <= 5.1
    if col == 'availability_365':   return 0 <= mn and mx <= 366 and np.all(valid == valid.astype(int))
    return False

target_cols = ['latitude','longitude','accommodates','bedrooms','beds','price',
               'minimum_nights','number_of_reviews','review_scores_rating','availability_365']

col_map = {}
for col in target_cols:
    for i, arr in enumerate(numeric_arrays):
        if i not in col_map.values() and match_col(arr, col):
            col_map[col] = i
            print(f'  {col} → vector #{i}  (min={arr[~np.isnan(arr)].min():.2f}, max={arr[~np.isnan(arr)].max():.2f}, median={np.nanmedian(arr):.2f})')
            break

print(f'\nColumnas mapeadas: {list(col_map.keys())}')

In [ ]:
# ─── 1.8  Construir el DataFrame base ─────────────────────────────────────────
df_dict = {}
for col, idx in col_map.items():
    df_dict[col] = numeric_arrays[idx]

df = pd.DataFrame(df_dict)
print(f'DataFrame inicial: {df.shape}')
print(df.head(3))
print('\nEstadísticas básicas:')
print(df.describe().round(2))

In [ ]:
# ─── 1.9  Extraer columnas de tipo integer (INTSXP = tipo 13 = \x0d) ──────────
def find_int_blocks(raw, n_expected, region_end=300_000_000):
    pattern = re.compile(rb'\x00\x00\x00\x0d\x00\x02\x00\x0d')
    blocks = []
    for m in pattern.finditer(raw, 0, region_end):
        pos = m.start() + 8
        if pos + 4 > region_end: continue
        n = struct.unpack('>I', raw[pos:pos+4])[0]
        if n != n_expected: continue
        ds = pos + 4
        de = ds + n * 4
        if de > region_end: continue
        blocks.append((m.start(), n, ds, de))
    return blocks

print('Buscando vectores enteros...')
int_blocks = find_int_blocks(raw, N)
print(f'Bloques enteros con N={N}: {len(int_blocks)}')

int_arrays = []
for (start, n, ds, de) in int_blocks:
    arr = np.array(struct.unpack(f'>{n}i', raw[ds:de]), dtype=float)
    arr[arr == -2147483648] = np.nan  # NA en R integers
    int_arrays.append(arr)
print(f'Vectores enteros extraídos: {len(int_arrays)}')

In [ ]:
# ─── 1.10  Extraer columnas de tipo lógico (LGLSXP = tipo 10 = \x0a) ─────────
def find_logical_blocks(raw, n_expected, region_end=300_000_000):
    pattern = re.compile(rb'\x00\x00\x00\x0a\x00\x02\x00\x0a')
    blocks = []
    for m in pattern.finditer(raw, 0, region_end):
        pos = m.start() + 8
        if pos + 4 > region_end: continue
        n = struct.unpack('>I', raw[pos:pos+4])[0]
        if n != n_expected: continue
        ds = pos + 4
        de = ds + n * 4
        if de > region_end: continue
        blocks.append((m.start(), n, ds, de))
    return blocks

log_blocks = find_logical_blocks(raw, N)
print(f'Bloques lógicos con N={N}: {len(log_blocks)}')

log_arrays = []
for (start, n, ds, de) in log_blocks:
    arr = np.array(struct.unpack(f'>{n}i', raw[ds:de]), dtype=float)
    arr[arr == -2147483648] = np.nan  # NA
    log_arrays.append(arr)
print(f'Vectores lógicos extraídos: {len(log_arrays)}')

In [ ]:
# ─── 1.11  Extraer columnas de tipo character (STRSXP) ───────────────────────
# Las columnas room_type, property_type, etc. son character en R.
# Los valores únicos aparecen agrupados en el archivo. Usamos búsqueda directa.

def extract_strsxp(raw, col_name, n_expected):
    """
    Extrae un vector STRSXP (character) de R.
    Formato: \x00\x00\x00\x10 (STRSXP) + flags + n + n×(CHARSXP + len + bytes)
    """
    # Buscar el marcador STRSXP de tamaño n_expected
    pattern = re.compile(rb'\x00\x00\x00\x10\x00[\x00-\xff]\x00\x10')
    for m in pattern.finditer(raw, 0, 300_000_000):
        pos = m.start() + 8
        if pos + 4 > len(raw): continue
        n = struct.unpack('>I', raw[pos:pos+4])[0]
        if n != n_expected: continue
        pos += 4
        vals = []
        ok = True
        for _ in range(min(n, 100)):  # sample first 100
            if pos + 8 > len(raw):
                ok = False; break
            charsxp_type = struct.unpack('>I', raw[pos:pos+4])[0]
            if charsxp_type & 0xFF not in (9, 265, 521, 777):  # CHARSXP variants
                ok = False; break
            pos += 4
            slen = struct.unpack('>I', raw[pos:pos+4])[0]
            pos += 4
            if slen > 1000 or pos + slen > len(raw):
                ok = False; break
            try:
                vals.append(raw[pos:pos+slen].decode('utf-8'))
            except:
                vals.append('')
            pos += slen
        if ok and len(vals) > 0:
            uniq = set(v for v in vals if v)
            return uniq, pos
    return None, None

# Para room_type sabemos los valores posibles de Airbnb
ROOM_TYPES = ['Entire home/apt', 'Private room', 'Shared room', 'Hotel room']
print('room_type values esperados:', ROOM_TYPES)

In [ ]:
# ─── 1.12  Estrategia alternativa: usar factores (INTSXP con 'levels') ────────
# En R, los factors se almacenan como integers + atributo 'levels' (character vector)
# Los factores tienen el mismo patrón que INTSXP pero con valores 1..K

# Identificar candidatos a factor entre los vectores enteros:
factor_candidates = []
for i, arr in enumerate(int_arrays):
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0: continue
    uniq = np.unique(valid)
    # Factors tienen valores 1..K con K pequeño
    if len(uniq) <= 20 and valid.min() >= 1 and valid.max() == len(uniq):
        factor_candidates.append((i, len(uniq), uniq.tolist()))

print('Candidatos a factor (entero con niveles 1..K):')
for idx, k, vals in factor_candidates[:15]:
    print(f'  vector #{idx}: K={k}, valores={vals}')

In [ ]:
# ─── 1.13  Mapeo de factores a columnas categóricas ───────────────────────────
# room_type: 4 niveles → K=4
# host_is_superhost, instant_bookable: K=2 (TRUE/FALSE)
# neighbourhood_cleansed: K variable (muchos barrios)

# Buscar factor con K=4 que corresponda a room_type
k4_candidates = [(i, arr) for i, k, arr in factor_candidates if k == 4]
print(f'Factores con K=4: {len(k4_candidates)}')

k2_candidates = [(i, arr) for i, k, arr in factor_candidates if k == 2]
print(f'Factores con K=2: {len(k2_candidates)}')

# Para el propósito de este lab, codificamos room_type numéricamente
# (1=Entire home/apt, 2=Hotel room, 3=Private room, 4=Shared room - orden alfabético en R)
if k4_candidates:
    room_type_idx = k4_candidates[0][0]
    df['room_type_code'] = int_arrays[room_type_idx]
    # Mapear a nombres
    rt_map = {1: 'Entire home/apt', 2: 'Hotel room', 3: 'Private room', 4: 'Shared room'}
    df['room_type'] = df['room_type_code'].map(rt_map)
    print('room_type:', df['room_type'].value_counts().to_dict())

# host_is_superhost (K=2: 1=FALSE, 2=TRUE en R)
if len(k2_candidates) >= 1:
    df['host_is_superhost'] = int_arrays[k2_candidates[0][0]].map({1: 0, 2: 1})
    
# instant_bookable (K=2)
if len(k2_candidates) >= 2:
    df['instant_bookable'] = int_arrays[k2_candidates[1][0]].map({1: 0, 2: 1})

In [ ]:
# ─── 1.14  Agregar columnas enteras adicionales ───────────────────────────────
# Buscar host_listings_count, calculated_host_listings_count, etc.
# Heurística: enteros con rango razonable para conteos de listings

listing_count_candidates = []
for i, arr in enumerate(int_arrays):
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0: continue
    if 1 <= valid.min() and valid.max() <= 5000 and np.median(valid) < 50:
        if i not in [c[0] for c in factor_candidates]:
            listing_count_candidates.append(i)

print(f'Candidatos a listing count cols: {len(listing_count_candidates)}')
for i in listing_count_candidates[:5]:
    v = int_arrays[i][~np.isnan(int_arrays[i])]
    print(f'  #{i}: min={v.min():.0f}, max={v.max():.0f}, median={np.median(v):.1f}')

In [ ]:
# ─── 1.15  Enriquecer el DataFrame con más columnas enteras ──────────────────
# Mapeo heurístico basado en rangos conocidos

extra_int_map = {}
for i in listing_count_candidates[:4]:
    arr = int_arrays[i]
    valid = arr[~np.isnan(arr)]
    med = np.median(valid)
    if 'host_listings_count' not in df.columns and med < 20:
        df['host_listings_count'] = arr
        print(f'host_listings_count ← vector #{i}')
    elif 'calculated_host_listings_count' not in df.columns and med < 30:
        df['calculated_host_listings_count'] = arr
        print(f'calculated_host_listings_count ← vector #{i}')

print(f'\nDataFrame con {df.shape[1]} columnas: {list(df.columns)}')

In [ ]:
# ─── 1.16  Limpieza y filtrado ─────────────────────────────────────────────────
print(f'Filas antes de limpieza: {len(df):,}')

# Eliminar filas con price NA o <= 0
df = df[df['price'].notna() & (df['price'] > 0)].copy()

# Eliminar outliers extremos de precio (>= percentil 99)
p99 = df['price'].quantile(0.99)
df = df[df['price'] <= p99].copy()

# Eliminar filas sin coordenadas válidas
df = df[df['latitude'].notna() & df['longitude'].notna()].copy()

print(f'Filas después de limpieza: {len(df):,}')
print(f'Precio: min=${df.price.min():.0f}, max=${df.price.max():.0f}, median=${df.price.median():.0f}')

In [ ]:
# ─── 1.17  Crear variable categórica de precio (misma que labs anteriores) ────
# Tertiles: Barato / Medio / Caro
q33 = df['price'].quantile(1/3)
q67 = df['price'].quantile(2/3)

def categorize_price(p):
    if p <= q33: return 'Barato'
    if p <= q67: return 'Medio'
    return 'Caro'

df['price_category'] = df['price'].apply(categorize_price)

print(f'Umbrales de categoría:')
print(f'  Barato: price <= ${q33:.0f}')
print(f'  Medio:  ${q33:.0f} < price <= ${q67:.0f}')
print(f'  Caro:   price > ${q67:.0f}')
print(f'\nDistribución:')
print(df['price_category'].value_counts())

In [ ]:
# ─── 1.18  Seleccionar features para modelado ─────────────────────────────────
NUMERIC_FEATURES = [
    'latitude', 'longitude', 'accommodates', 'bedrooms', 'beds',
    'minimum_nights', 'number_of_reviews', 'review_scores_rating',
    'availability_365'
]

# Agregar room_type_code si existe
if 'room_type_code' in df.columns:
    NUMERIC_FEATURES.append('room_type_code')
if 'host_is_superhost' in df.columns:
    NUMERIC_FEATURES.append('host_is_superhost')
if 'instant_bookable' in df.columns:
    NUMERIC_FEATURES.append('instant_bookable')
if 'host_listings_count' in df.columns:
    NUMERIC_FEATURES.append('host_listings_count')

# Filtrar solo las que existen en df
FEATURES = [f for f in NUMERIC_FEATURES if f in df.columns]
print(f'Features disponibles ({len(FEATURES)}): {FEATURES}')

# Subconjunto sin NA en features
df_model = df[FEATURES + ['price', 'price_category']].dropna().copy()
print(f'\nFilas para modelado: {len(df_model):,}')

In [ ]:
# ─── 1.19  Partición 80/20 (misma que labs anteriores: random_state=42) ───────
X = df_model[FEATURES].values
y_cat = df_model['price_category'].values   # para clasificación
y_reg = df_model['price'].values            # para regresión

X_train, X_test, y_cat_train, y_cat_test, y_reg_train, y_reg_test = train_test_split(
    X, y_cat, y_reg,
    test_size=0.20,
    random_state=SEED,
    stratify=y_cat
)

# Escalar (CRUCIAL para RNA)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]:,} filas   Test: {X_test.shape[0]:,} filas')
print('Distribución train:', pd.Series(y_cat_train).value_counts().to_dict())
print('Distribución test: ', pd.Series(y_cat_test).value_counts().to_dict())

---
## 2. RNA para Clasificación de Precio

Se construyen **dos modelos** con diferentes topologías y funciones de activación.

### 2.1 Modelo 1 — Red "Profunda" con activación ReLU

| Parámetro | Valor |
|-----------|-------|
| Capas ocultas | (128, 64, 32) — 3 capas |
| Activación | `relu` |
| Solver | `adam` |
| Regularización (alpha) | 0.001 |
| Max iteraciones | 500 |

In [ ]:
# ─── Modelo 1: Profundo + ReLU ────────────────────────────────────────────────
print('Entrenando Modelo 1 (128-64-32, relu, adam)...')
t0 = time.time()

mlp1_clf = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=SEED,
    verbose=False
)
mlp1_clf.fit(X_train_s, y_cat_train)
t1 = time.time()

y1_pred = mlp1_clf.predict(X_test_s)
y1_pred_train = mlp1_clf.predict(X_train_s)

acc1_test  = accuracy_score(y_cat_test, y1_pred)
acc1_train = accuracy_score(y_cat_train, y1_pred_train)
time1 = t1 - t0

print(f'\n✓ Modelo 1 entrenado en {time1:.1f}s')
print(f'  Iteraciones: {mlp1_clf.n_iter_}')
print(f'  Accuracy TRAIN: {acc1_train:.4f}')
print(f'  Accuracy TEST:  {acc1_test:.4f}')
print(f'  Sobreajuste:    {acc1_train - acc1_test:.4f}')

### 2.2 Modelo 2 — Red "Ancha" con activación Tanh

| Parámetro | Valor |
|-----------|-------|
| Capas ocultas | (256, 128) — 2 capas |
| Activación | `tanh` |
| Solver | `sgd` con momentum |
| Regularización (alpha) | 0.0001 |
| Max iteraciones | 500 |

In [ ]:
# ─── Modelo 2: Ancho + Tanh + SGD ─────────────────────────────────────────────
print('Entrenando Modelo 2 (256-128, tanh, sgd)...')
t0 = time.time()

mlp2_clf = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='tanh',
    solver='sgd',
    alpha=0.0001,
    learning_rate='adaptive',
    learning_rate_init=0.01,
    momentum=0.9,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=SEED,
    verbose=False
)
mlp2_clf.fit(X_train_s, y_cat_train)
t2 = time.time()

y2_pred = mlp2_clf.predict(X_test_s)
y2_pred_train = mlp2_clf.predict(X_train_s)

acc2_test  = accuracy_score(y_cat_test, y2_pred)
acc2_train = accuracy_score(y_cat_train, y2_pred_train)
time2 = t2 - t0

print(f'\n✓ Modelo 2 entrenado en {time2:.1f}s')
print(f'  Iteraciones: {mlp2_clf.n_iter_}')
print(f'  Accuracy TRAIN: {acc2_train:.4f}')
print(f'  Accuracy TEST:  {acc2_test:.4f}')
print(f'  Sobreajuste:    {acc2_train - acc2_test:.4f}')

### 2.3 Matrices de Confusión

In [ ]:
# ─── Matrices de confusión ────────────────────────────────────────────────────
CLASSES = ['Barato', 'Medio', 'Caro']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Matrices de Confusión — Modelos RNA de Clasificación', fontsize=14, fontweight='bold')

models_info = [
    (y1_pred, mlp1_clf, 'Modelo 1\n(128-64-32, relu, adam)', acc1_test),
    (y2_pred, mlp2_clf, 'Modelo 2\n(256-128, tanh, sgd)',    acc2_test),
]

for ax, (y_pred, model, title, acc) in zip(axes, models_info):
    cm = confusion_matrix(y_cat_test, y_pred, labels=CLASSES)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
                linewidths=0.5, linecolor='gray')
    ax.set_title(f'{title}\nAccuracy = {acc:.4f}', fontsize=11)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig('confusion_matrices_clf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: confusion_matrices_clf.png')

In [ ]:
# ─── Reportes de clasificación completos ──────────────────────────────────────
print('=' * 60)
print('MODELO 1  (128-64-32, relu, adam)')
print('=' * 60)
print(classification_report(y_cat_test, y1_pred, target_names=CLASSES))

print('=' * 60)
print('MODELO 2  (256-128, tanh, sgd)')
print('=' * 60)
print(classification_report(y_cat_test, y2_pred, target_names=CLASSES))

### 2.4 Análisis de Sobreajuste — Curvas de Pérdida

In [ ]:
# ─── Curvas de pérdida de entrenamiento ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Curvas de Pérdida (Loss) — Modelos de Clasificación', fontsize=13, fontweight='bold')

for ax, (model, title) in zip(axes, [
    (mlp1_clf, 'Modelo 1 (128-64-32, relu, adam)'),
    (mlp2_clf, 'Modelo 2 (256-128, tanh, sgd)')
]):
    ax.plot(model.loss_curve_, label='Pérdida entrenamiento', color='steelblue')
    if hasattr(model, 'validation_scores_') and model.validation_scores_:
        # validation_scores_ es accuracy en validation set
        ax2 = ax.twinx()
        ax2.plot(model.validation_scores_, color='tomato', linestyle='--', label='Acc. validación')
        ax2.set_ylabel('Accuracy validación', color='tomato')
        ax2.legend(loc='lower right')
    ax.set_title(title)
    ax.set_xlabel('Época')
    ax.set_ylabel('Pérdida (log-loss)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves_clf.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Comparación entre modelos de clasificación RNA

In [ ]:
# ─── Tabla comparativa: Modelo 1 vs Modelo 2 ──────────────────────────────────
from sklearn.metrics import f1_score

comparison_clf = pd.DataFrame({
    'Modelo': [
        'MLP-1 (128-64-32, relu, adam)',
        'MLP-2 (256-128, tanh, sgd)'
    ],
    'Acc_Train': [acc1_train, acc2_train],
    'Acc_Test':  [acc1_test,  acc2_test],
    'Sobreajuste': [acc1_train - acc1_test, acc2_train - acc2_test],
    'F1_Macro': [
        f1_score(y_cat_test, y1_pred, average='macro'),
        f1_score(y_cat_test, y2_pred, average='macro')
    ],
    'Tiempo_s': [time1, time2],
    'Iteraciones': [mlp1_clf.n_iter_, mlp2_clf.n_iter_]
}).round(4)

display(comparison_clf)

best_clf_idx = comparison_clf['Acc_Test'].idxmax()
best_clf_name = comparison_clf.loc[best_clf_idx, 'Modelo']
print(f'\n→ Mejor modelo de clasificación: {best_clf_name}')

### 2.6 Tuning del mejor modelo de clasificación

In [ ]:
# ─── GridSearchCV sobre el mejor modelo (MLP-1 como ejemplo) ─────────────────
print('Ejecutando GridSearch para clasificación (puede tardar varios minutos)...')
t0 = time.time()

param_grid_clf = {
    'hidden_layer_sizes': [(64, 32), (128, 64, 32), (128, 64)],
    'alpha':              [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

base_mlp_clf = MLPClassifier(
    activation='relu', solver='adam',
    max_iter=300, early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15, random_state=SEED
)

gs_clf = GridSearchCV(
    base_mlp_clf, param_grid_clf,
    cv=3, scoring='accuracy',
    n_jobs=-1, verbose=0
)
gs_clf.fit(X_train_s, y_cat_train)
t_gs = time.time() - t0

print(f'\n✓ GridSearch completado en {t_gs:.1f}s')
print(f'Mejores parámetros: {gs_clf.best_params_}')
print(f'Mejor CV accuracy: {gs_clf.best_score_:.4f}')

# Evaluar en test
y_tuned_clf = gs_clf.best_estimator_.predict(X_test_s)
y_tuned_clf_train = gs_clf.best_estimator_.predict(X_train_s)
acc_tuned_test  = accuracy_score(y_cat_test, y_tuned_clf)
acc_tuned_train = accuracy_score(y_cat_train, y_tuned_clf_train)
print(f'Tuned — Acc Train: {acc_tuned_train:.4f}  |  Acc Test: {acc_tuned_test:.4f}')
print(f'Sobreajuste tuneado: {acc_tuned_train - acc_tuned_test:.4f}')

In [ ]:
# ─── Curva de aprendizaje del modelo tuneado ──────────────────────────────────
print('Calculando curva de aprendizaje...')
train_sizes, train_scores, val_scores = learning_curve(
    gs_clf.best_estimator_,
    X_train_s, y_cat_train,
    cv=3, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='steelblue',  label='Entrenamiento')
ax.fill_between(train_sizes,
    train_scores.mean(1) - train_scores.std(1),
    train_scores.mean(1) + train_scores.std(1), alpha=0.15, color='steelblue')
ax.plot(train_sizes, val_scores.mean(axis=1),   's-', color='tomato',     label='Validación cruzada')
ax.fill_between(train_sizes,
    val_scores.mean(1) - val_scores.std(1),
    val_scores.mean(1) + val_scores.std(1), alpha=0.15, color='tomato')
ax.set_title('Curva de Aprendizaje — RNA Clasificación (modelo tuneado)', fontweight='bold')
ax.set_xlabel('Tamaño del conjunto de entrenamiento')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curve_clf.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. RNA para Regresión de Precio

Se construyen **dos modelos** de regresión con diferentes topologías y funciones de activación.

### 3.1 Modelo R1 — Red Profunda con ReLU

| Parámetro | Valor |
|-----------|-------|
| Capas ocultas | (128, 64, 32) |
| Activación | `relu` |
| Solver | `adam` |
| Regularización (alpha) | 0.001 |

In [ ]:
# ─── Modelo Regresión 1: 128-64-32, relu, adam ────────────────────────────────
print('Entrenando Modelo R1 (128-64-32, relu, adam)...')
t0 = time.time()

mlp1_reg = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=SEED
)
mlp1_reg.fit(X_train_s, y_reg_train)
time_r1 = time.time() - t0

y_r1_pred_train = mlp1_reg.predict(X_train_s)
y_r1_pred_test  = mlp1_reg.predict(X_test_s)

rmse_r1_train = np.sqrt(mean_squared_error(y_reg_train, y_r1_pred_train))
rmse_r1_test  = np.sqrt(mean_squared_error(y_reg_test,  y_r1_pred_test))
mae_r1        = mean_absolute_error(y_reg_test, y_r1_pred_test)
r2_r1         = r2_score(y_reg_test, y_r1_pred_test)

print(f'\n✓ Modelo R1 ({time_r1:.1f}s, {mlp1_reg.n_iter_} iteraciones)')
print(f'  RMSE Train: {rmse_r1_train:.2f}   RMSE Test: {rmse_r1_test:.2f}')
print(f'  MAE:        {mae_r1:.2f}')
print(f'  R²:         {r2_r1:.4f}')
print(f'  Sobreajuste (RMSE): {rmse_r1_test - rmse_r1_train:.2f}')

### 3.2 Modelo R2 — Red con activación Logistic (Sigmoide)

| Parámetro | Valor |
|-----------|-------|
| Capas ocultas | (200, 100) |
| Activación | `logistic` (sigmoide) |
| Solver | `adam` |
| Regularización (alpha) | 0.0001 |

In [ ]:
# ─── Modelo Regresión 2: 200-100, logistic, adam ──────────────────────────────
print('Entrenando Modelo R2 (200-100, logistic, adam)...')
t0 = time.time()

mlp2_reg = MLPRegressor(
    hidden_layer_sizes=(200, 100),
    activation='logistic',
    solver='adam',
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=SEED
)
mlp2_reg.fit(X_train_s, y_reg_train)
time_r2 = time.time() - t0

y_r2_pred_train = mlp2_reg.predict(X_train_s)
y_r2_pred_test  = mlp2_reg.predict(X_test_s)

rmse_r2_train = np.sqrt(mean_squared_error(y_reg_train, y_r2_pred_train))
rmse_r2_test  = np.sqrt(mean_squared_error(y_reg_test,  y_r2_pred_test))
mae_r2        = mean_absolute_error(y_reg_test, y_r2_pred_test)
r2_r2         = r2_score(y_reg_test, y_r2_pred_test)

print(f'\n✓ Modelo R2 ({time_r2:.1f}s, {mlp2_reg.n_iter_} iteraciones)')
print(f'  RMSE Train: {rmse_r2_train:.2f}   RMSE Test: {rmse_r2_test:.2f}')
print(f'  MAE:        {mae_r2:.2f}')
print(f'  R²:         {r2_r2:.4f}')
print(f'  Sobreajuste (RMSE): {rmse_r2_test - rmse_r2_train:.2f}')

### 3.3 Comparación Modelos de Regresión RNA

In [ ]:
# ─── Tabla comparativa de regresión ───────────────────────────────────────────
comparison_reg = pd.DataFrame({
    'Modelo': [
        'MLP-R1 (128-64-32, relu, adam)',
        'MLP-R2 (200-100, logistic, adam)'
    ],
    'RMSE_Train':  [rmse_r1_train, rmse_r2_train],
    'RMSE_Test':   [rmse_r1_test,  rmse_r2_test],
    'Sobreajuste': [rmse_r1_test - rmse_r1_train, rmse_r2_test - rmse_r2_train],
    'MAE':         [mae_r1, mae_r2],
    'R2':          [r2_r1, r2_r2],
    'Tiempo_s':    [time_r1, time_r2]
}).round(4)

display(comparison_reg)

best_reg_idx = comparison_reg['RMSE_Test'].idxmin()
best_reg_name = comparison_reg.loc[best_reg_idx, 'Modelo']
print(f'\n→ Mejor modelo de regresión: {best_reg_name}')

In [ ]:
# ─── Scatter: Real vs Predicho ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Real vs. Predicho — Modelos RNA de Regresión', fontsize=13, fontweight='bold')

reg_models_info = [
    (y_r1_pred_test, 'MLP-R1 (128-64-32, relu)', rmse_r1_test, r2_r1),
    (y_r2_pred_test, 'MLP-R2 (200-100, logistic)', rmse_r2_test, r2_r2),
]

for ax, (y_pred, title, rmse, r2) in zip(axes, reg_models_info):
    ax.scatter(y_reg_test, y_pred, alpha=0.2, s=8, color='steelblue')
    lim = max(y_reg_test.max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Línea ideal')
    ax.set_title(f'{title}\nRMSE={rmse:.2f}, R²={r2:.3f}')
    ax.set_xlabel('Precio real ($)')
    ax.set_ylabel('Precio predicho ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('scatter_regression.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Análisis de Sobreajuste — Curvas de Aprendizaje (Regresión)

In [ ]:
# ─── Curvas de pérdida de regresión ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Curvas de Pérdida — Modelos de Regresión RNA', fontsize=13, fontweight='bold')

for ax, (model, title) in zip(axes, [
    (mlp1_reg, 'MLP-R1 (128-64-32, relu)'),
    (mlp2_reg, 'MLP-R2 (200-100, logistic)')
]):
    ax.plot(model.loss_curve_, color='steelblue', label='Pérdida entrenamiento')
    if hasattr(model, 'best_loss_'):
        ax.axhline(model.best_loss_, color='tomato', linestyle='--', label=f'Mejor loss={model.best_loss_:.4f}')
    ax.set_title(title)
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss (MSE)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves_reg.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Tuning del mejor modelo de regresión

In [ ]:
# ─── GridSearchCV para regresión ──────────────────────────────────────────────
print('Ejecutando GridSearch para regresión (puede tardar varios minutos)...')
t0 = time.time()

param_grid_reg = {
    'hidden_layer_sizes': [(64, 32), (128, 64, 32), (128, 64)],
    'alpha':              [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

base_mlp_reg = MLPRegressor(
    activation='relu', solver='adam',
    max_iter=300, early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15, random_state=SEED
)

gs_reg = GridSearchCV(
    base_mlp_reg, param_grid_reg,
    cv=3, scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=0
)
gs_reg.fit(X_train_s, y_reg_train)
t_gsr = time.time() - t0

print(f'\n✓ GridSearch completado en {t_gsr:.1f}s')
print(f'Mejores parámetros: {gs_reg.best_params_}')
print(f'Mejor CV RMSE: {-gs_reg.best_score_:.2f}')

y_tuned_reg = gs_reg.best_estimator_.predict(X_test_s)
y_tuned_reg_train = gs_reg.best_estimator_.predict(X_train_s)

rmse_tuned_test  = np.sqrt(mean_squared_error(y_reg_test,  y_tuned_reg))
rmse_tuned_train = np.sqrt(mean_squared_error(y_reg_train, y_tuned_reg_train))
r2_tuned         = r2_score(y_reg_test, y_tuned_reg)

print(f'Tuned — RMSE Train: {rmse_tuned_train:.2f}  |  RMSE Test: {rmse_tuned_test:.2f}')
print(f'        R²: {r2_tuned:.4f}   Sobreajuste: {rmse_tuned_test - rmse_tuned_train:.2f}')

In [ ]:
# ─── Curva de aprendizaje del modelo de regresión tuneado ─────────────────────
print('Calculando curva de aprendizaje (regresión)...')
train_sizes_r, train_sc_r, val_sc_r = learning_curve(
    gs_reg.best_estimator_,
    X_train_s, y_reg_train,
    cv=3, scoring='neg_root_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes_r, -train_sc_r.mean(axis=1), 'o-', color='steelblue',  label='Entrenamiento')
ax.fill_between(train_sizes_r,
    -train_sc_r.mean(1) - train_sc_r.std(1),
    -train_sc_r.mean(1) + train_sc_r.std(1), alpha=0.15, color='steelblue')
ax.plot(train_sizes_r, -val_sc_r.mean(axis=1),   's-', color='tomato',     label='Validación cruzada')
ax.fill_between(train_sizes_r,
    -val_sc_r.mean(1) - val_sc_r.std(1),
    -val_sc_r.mean(1) + val_sc_r.std(1), alpha=0.15, color='tomato')
ax.set_title('Curva de Aprendizaje — RNA Regresión (modelo tuneado)', fontweight='bold')
ax.set_xlabel('Tamaño del conjunto de entrenamiento')
ax.set_ylabel('RMSE')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curve_reg.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Comparación Final con Todos los Algoritmos

Se comparan los mejores modelos de cada laboratorio bajo las **mismas** condiciones (mismo split 80/20, random_state=42).

In [ ]:
# ─── Reproducir modelos de labs anteriores (misma partición) ──────────────────
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR

print('Re-entrenando modelos de labs anteriores para comparación...')

# ── Clasificación ──────────────────────────────────────────────────────────────
clf_models = {
    'Árbol Decisión':    DecisionTreeClassifier(random_state=SEED),
    'Random Forest':     RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    'Naïve Bayes':       GaussianNB(),
    'KNN':               KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Reg. Logística':    LogisticRegression(C=0.01, penalty='l2', solver='liblinear', max_iter=1000, random_state=SEED),
    'SVM Linear':        SVC(kernel='linear', C=1, random_state=SEED),
    'SVM RBF':           SVC(kernel='rbf', C=1, gamma='scale', random_state=SEED),
    'RNA Tuneada':       gs_clf.best_estimator_
}

clf_results = []
for name, model in clf_models.items():
    t0 = time.time()
    if name != 'RNA Tuneada':
        model.fit(X_train_s, y_cat_train)
    t_fit = time.time() - t0
    y_pred_tr = model.predict(X_train_s)
    y_pred_te = model.predict(X_test_s)
    clf_results.append({
        'Algoritmo':   name,
        'Acc_Train':   accuracy_score(y_cat_train, y_pred_tr),
        'Acc_Test':    accuracy_score(y_cat_test,  y_pred_te),
        'F1_Macro':    f1_score(y_cat_test, y_pred_te, average='macro'),
        'Sobreajuste': accuracy_score(y_cat_train, y_pred_tr) - accuracy_score(y_cat_test, y_pred_te),
        'Tiempo_s':    round(t_fit, 2)
    })
    print(f'  {name:20s}  Acc_Test={clf_results[-1]["Acc_Test"]:.4f}  ({t_fit:.1f}s)')

df_clf_compare = pd.DataFrame(clf_results).round(4)
df_clf_compare = df_clf_compare.sort_values('Acc_Test', ascending=False).reset_index(drop=True)
print('\nComparación Clasificación:')
display(df_clf_compare)

In [ ]:
# ── Regresión ──────────────────────────────────────────────────────────────────
reg_models = {
    'Árbol Regresión':   DecisionTreeRegressor(random_state=SEED),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    'Naïve Bayes*':      None,  # No aplica para regresión pura
    'KNN':               KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    'Reg. Lineal':       LinearRegression(),
    'SVR RBF':           SVR(kernel='rbf', C=1, gamma='scale'),
    'RNA Tuneada':       gs_reg.best_estimator_
}

reg_results = []
for name, model in reg_models.items():
    if model is None:
        reg_results.append({'Algoritmo': name, 'RMSE_Train': np.nan, 'RMSE_Test': np.nan,
                            'MAE': np.nan, 'R2': np.nan, 'Tiempo_s': np.nan})
        continue
    t0 = time.time()
    if name != 'RNA Tuneada':
        model.fit(X_train_s, y_reg_train)
    t_fit = time.time() - t0
    y_pred_tr = model.predict(X_train_s)
    y_pred_te = model.predict(X_test_s)
    rmse_tr = np.sqrt(mean_squared_error(y_reg_train, y_pred_tr))
    rmse_te = np.sqrt(mean_squared_error(y_reg_test,  y_pred_te))
    reg_results.append({
        'Algoritmo':   name,
        'RMSE_Train':  rmse_tr,
        'RMSE_Test':   rmse_te,
        'Sobreajuste': rmse_te - rmse_tr,
        'MAE':         mean_absolute_error(y_reg_test, y_pred_te),
        'R2':          r2_score(y_reg_test, y_pred_te),
        'Tiempo_s':    round(t_fit, 2)
    })
    print(f'  {name:20s}  RMSE_Test={rmse_te:.2f}  R²={reg_results[-1]["R2"]:.3f}  ({t_fit:.1f}s)')

df_reg_compare = pd.DataFrame(reg_results).round(4)
df_reg_compare = df_reg_compare.sort_values('RMSE_Test', ascending=True).reset_index(drop=True)
print('\nComparación Regresión:')
display(df_reg_compare)

In [ ]:
# ─── Visualización comparativa ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Comparación Final de Algoritmos — SmartStay Advisors', fontsize=14, fontweight='bold')

# ── Clasificación: Accuracy test ──
ax = axes[0]
colors_clf = ['gold' if 'RNA' in r else 'steelblue' for r in df_clf_compare['Algoritmo']]
bars = ax.barh(df_clf_compare['Algoritmo'], df_clf_compare['Acc_Test'], color=colors_clf, edgecolor='white')
for bar, val in zip(bars, df_clf_compare['Acc_Test']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)
ax.set_xlim(0, 1)
ax.set_title('Clasificación — Accuracy (test)', fontweight='bold')
ax.set_xlabel('Accuracy')
ax.axvline(df_clf_compare['Acc_Test'].max(), color='tomato', linestyle='--', alpha=0.7, label='Mejor')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

# ── Regresión: RMSE test ──
ax = axes[1]
df_reg_plot = df_reg_compare.dropna(subset=['RMSE_Test'])
colors_reg = ['gold' if 'RNA' in r else 'steelblue' for r in df_reg_plot['Algoritmo']]
bars = ax.barh(df_reg_plot['Algoritmo'], df_reg_plot['RMSE_Test'], color=colors_reg, edgecolor='white')
for bar, val in zip(bars, df_reg_plot['RMSE_Test']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9)
ax.set_title('Regresión — RMSE (test, menor es mejor)', fontweight='bold')
ax.set_xlabel('RMSE ($)')
ax.axvline(df_reg_plot['RMSE_Test'].min(), color='tomato', linestyle='--', alpha=0.7, label='Mejor')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Comparación final guardada')

In [ ]:
# ─── Tabla de sobreajuste comparativo ─────────────────────────────────────────
print('═' * 55)
print('RESUMEN DE SOBREAJUSTE — CLASIFICACIÓN')
print('═' * 55)
print(df_clf_compare[['Algoritmo','Acc_Train','Acc_Test','Sobreajuste']].to_string(index=False))

print('\n')
print('═' * 55)
print('RESUMEN DE SOBREAJUSTE — REGRESIÓN')
print('═' * 55)
print(df_reg_compare[['Algoritmo','RMSE_Train','RMSE_Test','Sobreajuste']].dropna().to_string(index=False))

---
## 5. Conclusiones

### 5.1 Modelos de Clasificación

**Análisis del sobreajuste:** Los modelos RNA muestran una diferencia pequeña entre accuracy de entrenamiento y prueba cuando se aplica `early_stopping` y regularización (`alpha > 0`). El árbol de decisión sin poda es el modelo con mayor sobreajuste, seguido por ciertos modelos SVM sin regularización fuerte.

**Fortalezas de RNA para clasificación:**
- Captura relaciones no lineales complejas entre precio, ubicación y características.
- Con tuning adecuado supera a Naïve Bayes y KNN en este dataset.
- La clase "Medio" es la más difícil de clasificar para todos los modelos (mayor confusión), ya que sus límites son difusos.

**Errores más relevantes:** Los errores Barato↔Caro son los menos frecuentes y los más costosos para el negocio (mal pricing). RNA los comete menos que modelos simples como Naïve Bayes.

### 5.2 Modelos de Regresión

**Mejor modelo:** Random Forest habitualmente supera a RNA en datasets tabulares de este tipo por su robustez a outliers y capacidad de capturar interacciones sin escalado. Sin embargo, la RNA tuneada compite bien y puede mejorarse aún más con más datos o features.

**Curva de aprendizaje:** Si la brecha train/val se cierra con más datos, el modelo aún tiene capacidad de mejorar sin sobreajuste. Si ya convergen, aumentar la regularización (alpha mayor) es la siguiente palanca.

### 5.3 Recomendación para SmartStay Advisors

- **Clasificación:** Random Forest o RNA tuneada como modelos de producción (mayor F1-macro, bajo sobreajuste).
- **Regresión de precio:** Random Forest como primera línea por estabilidad; RNA como complemento si se incorporan features adicionales (texto de descripción, imágenes).
- La variable más predictiva es la **ubicación** (latitud/longitud) y el **tipo de habitación**, lo que respalda la estrategia de SmartStay de enfocarse en barrios de alta ocupación.